# Data Cleaning and Pre-processing
The set of steps used to prepare raw data so it can be reliably used for training or evaluating machine learning models. This includes fixing errors, handling missing values, removing duplicates, encoding categorical data, and detecting outliers. Good preprocessing improves model accuracy, reduces noise, and ensures the dataset represents the real problem correctly.

- Data collected
- Data needs to be processed into a useable format

- Fix data types
- Parse dates
- Remove duplicates
- Light missing value marking (but not full imputation)
- Basic text cleaning
- Basic filtering of corrupted rows
- Remove anomalies

This notebook is designed to process the many html pages scraped from basketball-reference. Here we use the beautiful soup library to read the html files and grab the relevant information we are looking for. We can do this by looking for certain classes in the html code.

In [1]:
import os
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
SCORE_DIR = "data/scores"

In [3]:
box_scores = os.listdir(SCORE_DIR)

In [4]:
box_scores = [os.path.join(SCORE_DIR, f) for f in box_scores if f.endswith(".html")]

In [5]:
def parse_html(box_score):
    html = None
    # Try UTF-8 first
    try:
        with open(box_score, 'r', encoding="utf-8") as f:
            html = f.read()
    except UnicodeDecodeError:
        # Try cp1252 (Windows-1252) if UTF-8 fails
        try:
            with open(box_score, 'r', encoding="cp1252") as f:
                html = f.read()
        except UnicodeDecodeError:
            print(f"⚠️ Skipping file due to encoding error: {box_score}")
            return None

    if not html or html.strip() == "":
        print(f"⚠️ Skipping empty file: {box_score}")
        return None

    soup = BeautifulSoup(html, "html.parser")
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup

In [6]:
def read_line_score(soup):
    try:
        line_score = pd.read_html(str(soup), attrs={"id": "line_score"})[0]
    except (ValueError, IndexError):
        # ValueError: no tables found, IndexError: empty result
        return None

    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols

    return line_score[["team", "total"]]

In [7]:
def read_stats(soup, team, stat):
    try:
        df = pd.read_html(
            str(soup), attrs={"id": f"box-{team}-game-{stat}"}, index_col=0
        )[0]
    except (ValueError, IndexError):
        return None

    return df.apply(pd.to_numeric, errors="coerce")

In [8]:
def read_season_info(soup):
    try:
        nav = soup.select("#bottom_nav_container")[0]
        hrefs = [a["href"] for a in nav.find_all("a")]
        season = os.path.basename(hrefs[1]).split("_")[0]
        return season
    except Exception:
        return None

In [9]:
base_cols = None
games = []

for box_score in box_scores:
    soup = parse_html(box_score)
    if soup is None:
        continue

    line_score = read_line_score(soup)
    if line_score is None:
        print(f"⚠️ Skipping file with no line_score: {box_score}")
        continue

    teams = list(line_score["team"])
    summaries = []

    for team in teams:
        basic = read_stats(soup, team, "basic")       #pulling the team's basic stats 
        advanced = read_stats(soup, team, "advanced") #pulling the team's more advanced stats
        if basic is None or advanced is None:
            print(f"⚠️ Skipping {box_score} due to missing stats for {team}")
            summaries = []
            break

        totals = pd.concat([basic.iloc[-1,:], advanced.iloc[-1,:]])
        totals.index = totals.index.str.lower()

        #Maximum statistic put up by one of the players
        #e.g. "GSW had a player put up the most points with 40pts, therefore pts_max=40"
        maxes = pd.concat([basic.iloc[:-1,:].max(), advanced.iloc[:-1,:].max()])
        maxes.index = maxes.index.str.lower() + "_max"

        summary = pd.concat([totals, maxes])

        if base_cols is None:
            base_cols = list(summary.index.drop_duplicates(keep="first"))
            base_cols = [b for b in base_cols if "bpm" not in b]

        summary = summary[base_cols]
        summaries.append(summary)

    if not summaries:
        continue  # skip incomplete game

    summary = pd.concat(summaries, axis=1).T
    game = pd.concat([summary, line_score], axis=1)

    game["home"] = [0, 1]
    game_opp = game.iloc[::-1].reset_index()
    game_opp.columns += "_opp"

    full_game = pd.concat([game, game_opp], axis=1)
    full_game["season"] = read_season_info(soup)
    full_game["date"] = pd.to_datetime(os.path.basename(box_score)[:8], format="%Y%m%d")
    full_game["won"] = full_game["total"] > full_game["total_opp"]

    games.append(full_game)

    if len(games) % 100 == 0:
        print(f"{len(games)} / {len(box_scores)}")


100 / 5240
200 / 5240
300 / 5240
400 / 5240
500 / 5240
600 / 5240
700 / 5240
800 / 5240
900 / 5240
1000 / 5240
1100 / 5240
⚠️ Skipping empty file: data/scores\201604010NYK.html
1200 / 5240
1300 / 5240
1400 / 5240
1500 / 5240
1600 / 5240
1700 / 5240
1800 / 5240
1900 / 5240
2000 / 5240
2100 / 5240
2200 / 5240
2300 / 5240
2400 / 5240
2500 / 5240
2600 / 5240
2700 / 5240
2800 / 5240
2900 / 5240
3000 / 5240
3100 / 5240
3200 / 5240
3300 / 5240
3400 / 5240
3500 / 5240
3600 / 5240
3700 / 5240
3800 / 5240
3900 / 5240
4000 / 5240
4100 / 5240
4200 / 5240
4300 / 5240
4400 / 5240
4500 / 5240
4600 / 5240
4700 / 5240
4800 / 5240
4900 / 5240
5000 / 5240
5100 / 5240
5200 / 5240


In [10]:
games_df = pd.concat(games, ignore_index=True)

In [11]:
games_df.to_csv("data/nba_games_raw.csv")

Parsing the html files takes a long time, therefore it is worth saving the progress of the parsed files and reloading them if we need to make changes to the rest of the preprocessing document.

In [12]:
games_df = pd.read_csv("data/nba_games_raw.csv",index_col=0)

Sometimes games include features which not all other games have, there may be games with 150 stats where some have 154 stats. This needs to be considered for model training since a model must have the same number of features to input from each training cycle.

In [13]:
len([g.shape[1] for g in games if g.shape[1] != 150])

5239

In [14]:
headers_list = [set(df.columns) for df in games]

In [15]:
common_headers = set.intersection(*headers_list)
print("Common headers:", len(common_headers))

Common headers: 150


In [16]:
# collect the "normal" headers (150)
baseline_headers = games[0].columns  # assuming the first game has 150

# look at games that don't have 150 headers
for i, g in enumerate(games):
    if g.shape[1] != 150:
        extras = set(g.columns) - set(baseline_headers)
        print(f"\nGame {i} has {g.shape[1]} headers")
        print("Extra headers:", extras)


Game 0 has 154 headers
Extra headers: set()

Game 1 has 154 headers
Extra headers: set()

Game 2 has 154 headers
Extra headers: set()

Game 3 has 154 headers
Extra headers: set()

Game 4 has 154 headers
Extra headers: set()

Game 5 has 154 headers
Extra headers: set()

Game 6 has 154 headers
Extra headers: set()

Game 7 has 154 headers
Extra headers: set()

Game 8 has 154 headers
Extra headers: set()

Game 9 has 154 headers
Extra headers: set()

Game 10 has 154 headers
Extra headers: set()

Game 11 has 154 headers
Extra headers: set()

Game 12 has 154 headers
Extra headers: set()

Game 13 has 154 headers
Extra headers: set()

Game 14 has 154 headers
Extra headers: set()

Game 15 has 154 headers
Extra headers: set()

Game 16 has 154 headers
Extra headers: set()

Game 17 has 154 headers
Extra headers: set()

Game 18 has 154 headers
Extra headers: set()

Game 19 has 154 headers
Extra headers: set()

Game 20 has 154 headers
Extra headers: set()

Game 21 has 154 headers
Extra headers: set(

Extra headers: set()

Game 1272 has 154 headers
Extra headers: set()

Game 1273 has 154 headers
Extra headers: set()

Game 1274 has 154 headers
Extra headers: set()

Game 1275 has 154 headers
Extra headers: set()

Game 1276 has 154 headers
Extra headers: set()

Game 1277 has 154 headers
Extra headers: set()

Game 1278 has 154 headers
Extra headers: set()

Game 1279 has 154 headers
Extra headers: set()

Game 1280 has 154 headers
Extra headers: set()

Game 1281 has 154 headers
Extra headers: set()

Game 1282 has 154 headers
Extra headers: set()

Game 1283 has 154 headers
Extra headers: set()

Game 1284 has 154 headers
Extra headers: set()

Game 1285 has 154 headers
Extra headers: set()

Game 1286 has 154 headers
Extra headers: set()

Game 1287 has 154 headers
Extra headers: set()

Game 1288 has 154 headers
Extra headers: set()

Game 1289 has 154 headers
Extra headers: set()

Game 1290 has 154 headers
Extra headers: set()

Game 1291 has 154 headers
Extra headers: set()

Game 1292 has 154 

Extra headers: set()

Game 2605 has 154 headers
Extra headers: set()

Game 2606 has 154 headers
Extra headers: set()

Game 2607 has 154 headers
Extra headers: set()

Game 2608 has 154 headers
Extra headers: set()

Game 2609 has 154 headers
Extra headers: set()

Game 2610 has 154 headers
Extra headers: set()

Game 2611 has 154 headers
Extra headers: set()

Game 2612 has 154 headers
Extra headers: set()

Game 2613 has 154 headers
Extra headers: set()

Game 2614 has 154 headers
Extra headers: set()

Game 2615 has 154 headers
Extra headers: set()

Game 2616 has 154 headers
Extra headers: set()

Game 2617 has 154 headers
Extra headers: set()

Game 2618 has 154 headers
Extra headers: set()

Game 2619 has 154 headers
Extra headers: set()

Game 2620 has 154 headers
Extra headers: set()

Game 2621 has 154 headers
Extra headers: set()

Game 2622 has 154 headers
Extra headers: set()

Game 2623 has 154 headers
Extra headers: set()

Game 2624 has 154 headers
Extra headers: set()

Game 2625 has 154 


Game 4134 has 154 headers
Extra headers: set()

Game 4135 has 154 headers
Extra headers: set()

Game 4136 has 154 headers
Extra headers: set()

Game 4137 has 154 headers
Extra headers: set()

Game 4138 has 154 headers
Extra headers: set()

Game 4139 has 154 headers
Extra headers: set()

Game 4140 has 154 headers
Extra headers: set()

Game 4141 has 154 headers
Extra headers: set()

Game 4142 has 154 headers
Extra headers: set()

Game 4143 has 154 headers
Extra headers: set()

Game 4144 has 154 headers
Extra headers: set()

Game 4145 has 154 headers
Extra headers: set()

Game 4146 has 154 headers
Extra headers: set()

Game 4147 has 154 headers
Extra headers: set()

Game 4148 has 154 headers
Extra headers: set()

Game 4149 has 154 headers
Extra headers: set()

Game 4150 has 154 headers
Extra headers: set()

Game 4151 has 154 headers
Extra headers: set()

Game 4152 has 154 headers
Extra headers: set()

Game 4153 has 154 headers
Extra headers: set()

Game 4154 has 154 headers
Extra headers

In [17]:
games_df.head()

,mp,mp.1,fg,fga,fg%,3p,3pa,3p%,ft,fta,...,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won
0,240.0,240.0,37.0,96.0,0.385,12.0,29.0,0.414,20.0,26.0,...,57.1,33.8,258.0,121.0,ATL,94,1,2016,2015-10-27,True
1,240.0,240.0,37.0,82.0,0.451,8.0,27.0,0.296,12.0,15.0,...,33.3,23.6,132.0,104.0,DET,106,0,2016,2015-10-27,False
2,240.0,240.0,38.0,94.0,0.404,9.0,29.0,0.310,10.0,17.0,...,53.2,34.6,162.0,104.0,CHI,97,1,2016,2015-10-27,False
3,240.0,240.0,37.0,87.0,0.425,7.0,19.0,0.368,16.0,23.0,...,30.4,29.0,138.0,105.0,CLE,95,0,2016,2015-10-27,True
4,240.0,240.0,35.0,83.0,0.422,6.0,18.0,0.333,19.0,27.0,...,69.4,43.7,206.0,104.0,GSW,111,1,2016,2015-10-27,False


In [18]:
def remove_cols(df, cols):
    for col in cols:
        del df[col]
    return df

In [19]:
def get_clean_df(df):
    df = df.sort_values("date")     #and sorting the data by date
    df = df.reset_index(drop=True)  #resetting the index and dropping the old index
    df = remove_cols(df, ["mp.1","mp_opp.1","index_opp"]) #removing unnecessary cols
    
    seasons = df['season'].unique()   # returns seasons in order of appearance
    last_three_seasons = seasons[-3:]   # get the last 3 seasons

    # Filter DataFrame
    df = df[df['season'].isin(last_three_seasons)]
    return df

Here we are trying to find out whether or not a given team will win their next game. We can train the data on the result we know from the next game by shifting the "won" column back by one and assigning it to target.

In [20]:
def add_target(team):
    team["target"] = team["won"].shift(-1)
    return team

When looking to shift the "won" column back to the previous, we must group the data by teams. We'd only shift a teams win back using the respective team's wins. For example, get all results where team_x is "GSW" and shift their games forward to find how they did in the next game (our target)

In [21]:
def group_teams(df):
    df = df.groupby("team",group_keys=False).apply(add_target).copy() #group by teams and shift corresponding "won" column back
    df["target"][pd.isnull(df["target"])] = 2   #If there isnt a next game change null value to 2
    df["target"] = df["target"].astype(int, errors="ignore")   #turn all target values to integer for model training

    return df

In [22]:
def remove_null_values(df):
    nulls = pd.isnull(df)
    nulls = nulls.sum()
    nulls = nulls[nulls > 0]
    
    valid_columns = df.columns[~df.columns.isin(nulls.index)]   #all columns which dont contain null values
    df = df[valid_columns].copy()
    
    return df

In [23]:
df = get_clean_df(games_df)
df = group_teams(df)
df = remove_null_values(df)

C:\Users\henXc\AppData\Local\Temp/ipykernel_14856/309807057.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["target"][pd.isnull(df["target"])] = 2   #If there isnt a next game change null value to 2


In [24]:
df = df.dropna() #Drop all null/na values

In [25]:
df.to_csv("data/nba_games.csv")